In [11]:
import torch as th

bs = 3
classes_num = 10

class MLP(th.nn.Module):
    """
    Simple model with method for loss and hand-written backprop.
    """

    def __init__(self) -> None:
        super(MLP, self).__init__()
        self.fc1 = th.nn.Linear(784, 100)
        self.relu = th.nn.ReLU()
        self.fc2 = th.nn.Linear(100, 10)

    
    def predict(self, x):
        self.z1 = self.fc1(x)
        self.a1 = self.relu(self.z1)
        return self.fc2(self.a1)

    def backward(self, X, error):
        z1_grad = (error @ self.fc2.state_dict()["weight"]) * (self.a1 > 0).float()
        fc1_weight_grad = z1_grad.t() @ X
        fc1_bias_grad = z1_grad.sum(0)
        fc2_weight_grad = error.t() @ self.a1
        fc2_bias_grad = error.sum(0)
        return fc1_weight_grad, fc1_bias_grad, fc2_weight_grad, fc2_bias_grad

    def softmax_cross_entropy_with_logits(self, logits, target, batch_size):
        probs = self.torch_ref.softmax(logits, dim=1)
        loss = -(target * self.torch_ref.log(probs)).sum(dim=1).mean()
        loss_grad = (probs - target) / batch_size
        return loss, loss_grad

    def accuracy(self, logits, targets, batch_size):
        pred = self.torch_ref.argmax(logits, dim=1)
        targets_idx = self.torch_ref.argmax(targets, dim=1)
        acc = pred.eq(targets_idx).sum().float() / batch_size
        return acc

    def forward(
    xs=th.randn(bs, 28 * 28),
    ys=th.nn.functional.one_hot(th.randint(0, classes_num, [bs]), classes_num),
    batch_size=th.tensor([bs]),
    lr=th.tensor([0.1])):

        # forward
        logits = predict(xs)

        # loss
        loss, loss_grad = model.softmax_cross_entropy_with_logits(
            logits, ys, batch_size
        )

        # backward
        grads = self.backward(xs, loss_grad)

        # SGD step
        updated_params = tuple(
            param - lr * grad for param, grad in zip(model.parameters(), grads)
        )

        # accuracy
        acc = self.accuracy(logits, ys, batch_size)

        # return things
        return (loss, acc, *updated_params)


In [12]:
ts = th.jit.script(MLP())
ts.code

UnsupportedNodeError: GeneratorExp aren't supported:
  File "/var/folders/gd/2d23t15d76gfvz06bpycmnwr0000gn/T/ipykernel_72438/2538287234.py", line 61
    
        # SGD step
        updated_params = tuple(
                              ~ <--- HERE
            param - lr * grad for param, grad in zip(model.parameters(), grads)
        )


In [17]:
load = th.jit.load("/Users/danielnugraha/Documents/bachelorarbeit/flower-swift-client/FlowerSDK/FlowerSDK/Model/syftModel.pt")
load.code

RuntimeError: [enforce fail at inline_container.cc:145] . PytorchStreamReader failed reading zip archive: failed finding central directory